In [189]:
import tkinter as tk
from tkinter import ttk, messagebox
import random

class DraggableWord(tk.Label):
    """
    Representa uma palavra arrastável na interface do jogo.
    Gerencia seu próprio estado de arrasto e interação visual.
    """
    def __init__(self, master, text, parent_app, is_correct_word=False, **kwargs):
        # Usando ttk.Label para melhor integração com temas
        super().__init__(master, text=text, bd=1, relief="raised",
                         padx=10, pady=5, font=("Segoe UI", 12, "bold"), cursor="hand2", **kwargs)
        
        self.word_text = text
        self.parent_app = parent_app  # Referência à instância da classe principal
        self.is_correct_word = is_correct_word # Flag para diferenciar palavras da frase de distratores
        
        # Cores de fundo diferenciadas para palavras corretas e distratores
        self.original_bg = "#E8F5E9" if is_correct_word else "#CFD8DC" # Verde claro para corretas, cinza azulado para distratores
        self.config(bg=self.original_bg, fg="#263238") # Cor do texto escura para contraste

        self._drag_start_x = 0
        self._drag_start_y = 0

        self.bind("<Button-1>", self._on_drag_start)
        self.bind("<B1-Motion>", self._on_drag_motion)
        self.bind("<ButtonRelease-1>", self._on_drag_release)

    def _on_drag_start(self, event):
        """Inicia o processo de arrasto, elevando a palavra e mudando seu visual."""
        self.lift()  # Traz a palavra para a frente
        self._drag_start_x = event.x
        self._drag_start_y = event.y
        self.config(relief="solid", bd=2) # Efeito visual de "levantado"

    def _on_drag_motion(self, event):
        """Move a palavra com o mouse, limitando o movimento à janela principal."""
        # Calcula a nova posição global da palavra
        x_root = self.winfo_rootx() + event.x - self._drag_start_x
        y_root = self.winfo_rooty() + event.y - self._drag_start_y

        # Converte a nova posição global para coordenadas relativas ao master atual
        # Isso permite que a palavra seja arrastada para fora do frame original
        x_new = x_root - self.master.winfo_rootx()
        y_new = y_root - self.master.winfo_rooty()

        # Garante que a palavra não saia completamente da janela principal (root)
        root_width = self.parent_app.root.winfo_width()
        root_height = self.parent_app.root.winfo_height()
        
        x_new_limited = max(0, min(x_new, root_width - self.winfo_width() - self.master.winfo_x()))
        y_new_limited = max(0, min(y_new, root_height - self.winfo_height() - self.master.winfo_y()))
        
        self.place(x=x_new_limited, y=y_new_limited)
        
        # Notifica o aplicativo pai para verificar colisões e destacar áreas
        self.parent_app.check_all_collisions()
        self.parent_app.highlight_assembly_area_on_hover(self)

    def _on_drag_release(self, event):
        """Finaliza o arrasto, reseta o visual e faz verificações finais."""
        self.config(relief="raised", bd=1) # Volta ao relevo normal
        self.parent_app.check_all_collisions() # Verifica colisão final
        self.parent_app.highlight_assembly_area_on_hover(self, reset=True) # Reseta o destaque da área de montagem
        
        # Se não houver colisão com outras palavras, reseta a cor para a original
        if not self.parent_app._is_colliding_with_any_other_word(self):
            self.config(bg=self.original_bg)


class LanguageGameApp:
    """
    Classe principal da aplicação do jogo de formação de frases.
    Gerencia o estado do jogo, níveis, UI e lógica.
    """
    # Definindo as frases do jogo e suas palavras distrativas
    PHRASES = [
        {"portugues": "Isso é uma caneta", "ingles": "this is a pen", "distractors": ["apple", "my", "go", "run", "dog"]},
        {"portugues": "Eu gosto de programar", "ingles": "I like to code", "distractors": ["read", "play", "sleep", "eat", "jump"]},
        {"portugues": "O sol está brilhando", "ingles": "the sun is shining", "distractors": ["moon", "stars", "cloud", "rain", "dark"]},
        {"portugues": "Ela está lendo um livro", "ingles": "she is reading a book", "distractors": ["writing", "singing", "dancing", "movie", "computer"]},
    ]
    
    def __init__(self, root):
        self.root = root
        self.root.title("Monte a Frase Correta!")
        self.root.geometry("950x680") # Aumenta um pouco mais a janela
        self.root.resizable(False, False) # Impede o redimensionamento para manter o layout
        self.root.configure(bg="#ECEFF1") # Fundo cinza claro para a janela principal

        self.current_level = 0
        self.attempts = 0
        self.current_phrase_data = {}
        self.draggable_words = [] # Lista para manter referências a todas as palavras arrastáveis

        self._setup_styles() # Configura estilos globais para ttk
        self._create_widgets() # Cria todos os elementos da interface
        self._load_level() # Carrega o primeiro nível do jogo

    def _setup_styles(self):
        """Configura estilos para widgets ttk para uma aparência moderna."""
        style = ttk.Style()
        style.theme_use('clam') # Um tema limpo e moderno

        # Estilos gerais para TFrame, TLabel e TButton
        style.configure('TFrame', background='#FFFFFF', borderwidth=0, relief="flat")
        style.configure('TLabel', background='#ECEFF1', font=("Segoe UI", 12), foreground="#333333")
        style.configure('Header.TLabel', font=("Segoe UI", 20, "bold"), foreground="#2C3E50", background="#ECEFF1")
        style.configure('SubHeader.TLabel', font=("Segoe UI", 14, "bold"), foreground="#444444", background="#ECEFF1")
        
        style.configure('Primary.TButton', font=("Segoe UI", 13, "bold"), padding=10, 
                        background="#3F51B5", foreground="white", relief="flat") # Azul primário
        style.map('Primary.TButton', background=[('active', '#303F9F')])
        
        style.configure('Secondary.TButton', font=("Segoe UI", 12), padding=10, 
                        background="#78909C", foreground="white", relief="flat") # Cinza secundário
        style.map('Secondary.TButton', background=[('active', '#607D8B')])

    def _create_widgets(self):
        """Cria e posiciona todos os widgets da interface do jogo."""
        
        # Título principal da aplicação
        ttk.Label(self.root, text="Monte a Frase em Inglês!", style='Header.TLabel').place(relx=0.5, y=30, anchor="center")


        # Rótulo da frase em português a ser traduzida
        self.portugues_label = ttk.Label(self.root, text="", font=("Segoe UI", 16, "italic"), foreground="#555555", background="#ECEFF1")
        self.portugues_label.place(x=50, y=70)

        # Rótulo de tentativas do usuário
        self.attempts_label = ttk.Label(self.root, text="Tentativas: 0", font=("Segoe UI", 12), foreground="#607D8B", background="#ECEFF1")
        self.attempts_label.place(x=780, y=70)

        # --- Área de Palavras Disponíveis (Pool) ---
        ttk.Label(self.root, text="Palavras disponíveis:", style='SubHeader.TLabel').place(x=50, y=120)
        self.words_pool_frame = ttk.Frame(self.root, width=850, height=400, relief="ridge", borderwidth=1)
        self.words_pool_frame.place(x=50, y=150)
        self.root.update_idletasks() # Garante que as dimensões do frame estejam prontas

        self.assembly_area_ui()

        # self.assembly_area = tk.Label(
        #     self.words_pool_frame, text="Monte a frase aqui ↓", bg="#FAFAFA",
        #     font=("Segoe UI", 12, "italic"), fg="#777", relief="solid", bd=1
        # )
        # self.assembly_area.place(x=50, y=230, width=700, height=100)

        

        # --- Botões de Ação ---
        self.check_button = ttk.Button(self.root, text="Verificar Frase", command=self._check_phrase_action, style='Primary.TButton')
        self.check_button.place(x=300, y=580, width=150, height=50)

        self.reset_button = ttk.Button(self.root, text="Resetar", command=self._reset_level, style='Secondary.TButton')
        self.reset_button.place(x=500, y=580, width=120, height=50)
        
        # --- Label de Resultado ---
        self.result_label = ttk.Label(self.root, text="", font=("Segoe UI", 18, "bold"), background="#ECEFF1")
        self.result_label.place(relx=0.5, y=540, anchor="center")

    def assembly_area_ui(self):
        self.assembly_area_frame = ttk.Frame(self.root, width=850, height=100, 
                                             relief="ridge", borderwidth=2)
        # self.assembly_area_frame.place(x=100, y=420, width=700, height=100)
        
        # Rótulo de dica dentro da área de montagem
        self.assembly_hint_label = ttk.Label(self.words_pool_frame, text="Monte a frase aqui ↓",
                                       font=("Segoe UI", 12, "italic"),
                                       background="#FAFAFA",
                                       foreground="#777", 
                                       relief="ridge", 
                                       border=1,
                                       justify="center",
                                       )
        self.assembly_hint_label.place(x=50, y=230, width=700, height=100)
        # self.assembly_hint_label.pack(expand=True, fill="both") # Centraliza o hint no frame

    def _load_level(self):
        """Carrega os dados da frase para o nível atual e inicializa o jogo."""
        if self.current_level >= len(self.PHRASES):
            messagebox.showinfo("Fim do Jogo", "Parabéns! Você completou todas as fases!")
            self.root.destroy()
            return

        self.current_phrase_data = self.PHRASES[self.current_level]
        self.correct_word_order = self.current_phrase_data["ingles"].split()
        
        self.portugues_label.config(text=f"Traduza: \"{self.current_phrase_data['portugues']}\"")
        self.attempts = 0
        self.attempts_label.config(text=f"Tentativas: {self.attempts}")
        self.result_label.config(text="")
        self.check_button.config(state="normal") # Garante que o botão esteja habilitado para o novo nível
        
        self._clear_words() # Remove palavras do nível anterior
        self._place_initial_words() # Posiciona as palavras do novo nível

    def _clear_words(self):
        """Destrói todas as palavras arrastáveis da tela."""
        for word_widget in self.draggable_words:
            word_widget.destroy()
        self.draggable_words.clear()

    def _place_initial_words(self):
        """Posiciona as palavras arrastáveis inicialmente de forma organizada no pool."""
        all_words_for_level = self.correct_word_order + self.current_phrase_data["distractors"]
        random.shuffle(all_words_for_level)

        words_per_row = 7 # Quantas palavras por linha no pool
        x_offset = 20
        y_offset = 20
        word_spacing_x = 120 # Espaçamento horizontal entre palavras
        word_spacing_y = 60 # Espaçamento vertical entre linhas de palavras

        for i, word_text in enumerate(all_words_for_level):
            row = i // words_per_row
            col = i % words_per_row
            
            x = x_offset + col * word_spacing_x
            y = y_offset + row * word_spacing_y
            
            # Verifica se a palavra é da frase correta para passar a flag
            is_correct = word_text in self.correct_word_order
            
            word_box = DraggableWord(self.words_pool_frame, word_text, self, is_correct_word=is_correct)
            word_box.place(x=x, y=y)
            self.draggable_words.append(word_box)

    def highlight_assembly_area_on_hover(self, current_word, reset=False):
        """
        Destaca a área de montagem da frase quando uma palavra arrastável
        está sobre ela, ou reseta o destaque.
        """
        if reset:
            # self.assembly_area_frame.config(relief="ridge", bd=2, highlightbackground="", highlightthickness=0)
            self.assembly_hint_label.config(text="Arraste as palavras para cá...", foreground="#90A4AE")
            return

        # Coordenadas da palavra e da área de montagem no sistema de coordenadas do ROOT
        word_x1_root = current_word.winfo_rootx()
        word_y1_root = current_word.winfo_rooty()
        word_x2_root = word_x1_root + current_word.winfo_width()
        word_y2_root = word_y1_root + current_word.winfo_height()


        assembly_x1_root = self.assembly_hint_label.winfo_rootx()
        assembly_y1_root = self.assembly_hint_label.winfo_rooty()
        assembly_x2_root = assembly_x1_root + self.assembly_hint_label.winfo_width()
        assembly_y2_root = assembly_y1_root + self.assembly_hint_label.winfo_height()

        # Verifica se há alguma sobreposição entre a palavra e a área de montagem
        # Usamos 50% da área da palavra para considerar uma sobreposição significativa
        overlap_x = max(0, min(word_x2_root, assembly_x2_root) - max(word_x1_root, assembly_x1_root))
        overlap_y = max(0, min(word_y2_root, assembly_y2_root) - max(word_y1_root, assembly_y1_root))
        
        word_area = current_word.winfo_width() * current_word.winfo_height()
        overlap_area = overlap_x * overlap_y

        # if word_area > 0 and (overlap_area / word_area) > 0.5: # Mais de 50% da palavra dentro da área
        #     self.assembly_hint_label.config(relief="solid", border=3, highlightbackground="#4CAF50", highlightthickness=2)
        #     self.assembly_hint_label.config(text="Solte aqui para montar!", foreground="#4CAF50")
        # else:
        #     self.assembly_hint_label.config(relief="ridge", border=2, highlightbackground="", highlightthickness=0)
        #     self.assembly_hint_label.config(text="Arraste as palavras para cá...", foreground="#90A4AE")

    def _is_colliding_with_any_other_word(self, target_word):
        """Verifica se uma palavra específica está colidindo com qualquer outra palavra arrastável."""
        # Itera sobre todas as palavras arrastáveis gerenciadas pelo aplicativo
        for other_word in self.draggable_words:
            if other_word != target_word and self._are_colliding(target_word, other_word):
                return True
        return False

    def check_all_collisions(self):
        """Verifica e atualiza as cores de todas as palavras baseadas em colisões."""
        for word_widget in self.draggable_words:
            if self._is_colliding_with_any_other_word(word_widget):
                word_widget.config(bg="red") # Alerta visual de colisão
            else:
                word_widget.config(bg=word_widget.original_bg) # Retorna à cor original

    def _are_colliding(self, widget1, widget2):
        """
        Verifica se dois widgets estão colidindo usando suas coordenadas globais no root.
        Mais robusto, pois não depende do master dos widgets.
        """
        x1, y1, w1, h1 = widget1.winfo_rootx(), widget1.winfo_rooty(), widget1.winfo_width(), widget1.winfo_height()
        x2, y2, w2, h2 = widget2.winfo_rootx(), widget2.winfo_rooty(), widget2.winfo_width(), widget2.winfo_height()

        # Retorna True se houver sobreposição nos eixos X e Y
        return not (x1 + w1 < x2 or x1 > x2 + w2 or
                    y1 + h1 < y2 or y1 > y2 + h2)

    def _is_word_in_assembly_area(self, word_widget):
        """
        Verifica se uma palavra está significativamente dentro da área de montagem,
        considerando uma sobreposição de mais de 50% da área da palavra.
        """
        # Coordenadas da palavra no sistema de coordenadas do ROOT
        word_x1_root = word_widget.winfo_rootx()
        word_y1_root = word_widget.winfo_rooty()
        word_x2_root = word_x1_root + word_widget.winfo_width()
        word_y2_root = word_y1_root + word_widget.winfo_height()

        # Coordenadas da área de montagem no sistema de coordenadas do ROOT
        assembly_x1_root = self.assembly_hint_label.winfo_rootx()
        assembly_y1_root = self.assembly_hint_label.winfo_rooty()
        assembly_x2_root = assembly_x1_root + self.assembly_hint_label.winfo_width()
        assembly_y2_root = assembly_y1_root + self.assembly_hint_label.winfo_height()

        # Calcula a área de sobreposição
        overlap_x = max(0, min(word_x2_root, assembly_x2_root) - max(word_x1_root, assembly_x1_root))
        overlap_y = max(0, min(word_y2_root, assembly_y2_root) - max(word_y1_root, assembly_y1_root))
        
        word_area = word_widget.winfo_width() * word_widget.winfo_height()
        overlap_area = overlap_x * overlap_y

        # Considera a palavra "dentro" se mais de 50% de sua área estiver na área de montagem
        return word_area > 0 and (overlap_area / word_area) > 0.5

    def _check_phrase_action(self):
        """Ação executada ao clicar no botão 'Verificar Frase'."""
        self.attempts += 1
        self.attempts_label.config(text=f"Tentativas: {self.attempts}")

        # Coleta apenas as palavras que estão na área de montagem
        assembled_words = []
        for word_widget in self.draggable_words:
            if self._is_word_in_assembly_area(word_widget):
                assembled_words.append(word_widget)

        # Ordena as palavras pela posição X (da esquerda para a direita)
        assembled_words.sort(key=lambda w: w.winfo_rootx())
        current_phrase = [word.word_text for word in assembled_words]

        if current_phrase == self.correct_word_order:
            self.result_label.config(text="✅ Correto! Parabéns!", foreground="#28A745") # Verde de sucesso
            self.check_button.config(state="disabled") # Desabilita o botão para evitar cliques repetidos
            messagebox.showinfo("Parabéns!", "Você montou a frase corretamente!")
            self.root.after(1500, self._next_level) # Espera um pouco e carrega o próximo nível
        else:
            self.result_label.config(text="❌ Incorreto! Tente novamente.", foreground="#DC3545") # Vermelho de erro

        print(f"Nível: {self.current_level + 1} | Tentativas: {self.attempts}")
        print("Frase esperada:", self.correct_word_order)
        print("Frase montada:", current_phrase)
        print("-" * 40)

    def _reset_level(self):
        """Reseta o nível atual, embaralhando e recolocando as palavras."""
        confirm = messagebox.askyesno("Resetar Nível", "Tem certeza que deseja resetar este nível?")
        if confirm:
            self.attempts = 0
            self.attempts_label.config(text=f"Tentativas: {self.attempts}")
            self.result_label.config(text="")
            self.check_button.config(state="normal") # Habilita o botão
            self._clear_words() # Remove as palavras existentes
            self._place_initial_words() # Posiciona novas palavras embaralhadas

    def _next_level(self):
        """Avança para o próximo nível do jogo."""
        self.current_level += 1
        self._load_level() # Carrega os dados do próximo nível

# Bloco principal de execução do aplicativo
if __name__ == "__main__":
    root = tk.Tk()
    app = LanguageGameApp(root)
    root.mainloop()

